In [6]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [7]:
class ScaledRFPDataset(Dataset):
    def __init__(self, pt_file_path, mu, sigma):
        data = torch.load(pt_file_path, weights_only=False)
        self.embeddings = data['embeddings']
        raw_log_prices = data['log_prices']
        self.targets = (raw_log_prices - mu) / sigma
        
    def __len__(self): return len(self.embeddings)
    def __getitem__(self, idx):
        return self.embeddings[idx], torch.tensor(self.targets[idx], dtype=torch.float32)
    
def tube_loss(y, mu1, mu2, t=0.95, r=0.5): 
    # Using the standard mathematically pure tube_loss for now
    lower = torch.min(mu1, mu2)
    upper = torch.max(mu1, mu2)
    
    loss = torch.zeros_like(y)
    loss[y > upper] = t * (y[y > upper] - upper[y > upper])
    loss[y < lower] = t * (lower[y < lower] - y[y < lower])

    mid = r * upper + (1 - r) * lower
    mask = (y >= lower) & (y <= upper)
    loss[mask & (y >= mid)] = (1 - t) * (upper[mask & (y >= mid)] - y[mask & (y >= mid)])
    loss[mask & (y < mid)]  = (1 - t) * (y[mask & (y < mid)] - lower[mask & (y < mid)])
    
    return loss.mean()


In [8]:
class CrossAttentionCategoryHead(nn.Module):
    def __init__(self, embed_dim=768, num_heads=4):
        # Note: 1536-D vector is actually two 768-D vectors (image + text) concatenated.
        super().__init__()
        
        # The Cross-Attention Layer: Text attends to Image
        self.cross_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        
        self.fc_head = nn.Sequential(
            nn.Linear(embed_dim * 2, 512), 
            nn.LayerNorm(512), 
            nn.GELU(), 
            nn.Dropout(0.3),
            nn.Linear(512, 256), 
            nn.GELU(), 
            nn.Dropout(0.2),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = x.float()
        
        img_emb = x[:, :768].unsqueeze(1)  
        txt_emb = x[:, 768:].unsqueeze(1) 

        # Cross Attention: Q = Image, K = Text, V = Text
        attn_out, _ = self.cross_attn(query=img_emb, key=txt_emb, value=txt_emb)
        
        attn_out = attn_out.squeeze(1)
        img_out = img_emb.squeeze(1)
        
        fused_features = torch.cat([img_out, attn_out], dim=1) # Shape: (Batch, 1536)
        
        return self.fc_head(fused_features)

In [9]:
def calculate_interval_score(y_true, lower, upper, alpha=0.05):
    """
    Calculates the Interval Score (IS) for 95% confidence intervals (alpha=0.05).
    y_true, lower, and upper should be PyTorch tensors.
    """

    width = upper - lower
    
    #Penalty if the true price is lower than the lower bound
    penalty_lower = (2.0 / alpha) * (lower - y_true)
    penalty_lower = torch.where(y_true < lower, penalty_lower, torch.zeros_like(width))
    
    #Penalty if the true price is higher than the upper bound
    penalty_upper = (2.0 / alpha) * (y_true - upper)
    penalty_upper = torch.where(y_true > upper, penalty_upper, torch.zeros_like(width))
    
    #final score is the width plus penalties for missing the target
    interval_score = width + penalty_lower + penalty_upper
    
    return interval_score.mean().item()

In [115]:
def train_cross_attention_model(split_dir="split_embeddings", epochs=15):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}\n")
    
    category_t_map = {
        "BIKES": 0.94, "BOOKS": 0.97, "CARS": 0.97, "CYCLE": 0.97,
        "FLAT": 0.96, "FRIDGES": 0.97, "GAMES": 0.97, "GAMESENTERTAINMENT": 0.96,
        "LAPTOP": 0.97, "MOBILE": 0.95, "PHONES": 0.96, "PRINTER": 0.95,
        "TV": 0.97, "WASHINGMACHINE": 0.94
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name, 0.95)
        alpha = 1.0 - t_val  # Required for Interval Score calculation
        
        print(f"\nCROSS-ATTENTION MODEL: {cat_name}\n{'-'*50}")
        
        # Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        # Initialize Cross-Attention Model
        model = CrossAttentionCategoryHead().to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        # Train
        model.train()
        for epoch in range(epochs):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                
                optimizer.zero_grad()
                out = model(x)
                loss = tube_loss(y, out[:,0], out[:,1], t=t_val, r=0.5)
                loss.backward()
                optimizer.step()

        model.eval()
        lowers, uppers, ys = [], [], []
        
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_device = x_batch.to(device)
                
                out = model(x_device)
                l = torch.min(out[:,0], out[:,1])
                u = torch.max(out[:,0], out[:,1])
                
                # Inverse Transform to real Rupees
                l_real = torch.exp(l * cat_sigma + cat_mu)
                u_real = torch.exp(u * cat_sigma + cat_mu)
                y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
                
                lowers.append(l_real)
                uppers.append(u_real)
                ys.append(y_true)
                
        l_preds = torch.cat(lowers)
        u_preds = torch.cat(uppers)
        y_trues = torch.cat(ys)
        
        # Metrics Calculations
        picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
        mpiw = (u_preds - l_preds).mean().item()
        mid_preds = (l_preds + u_preds) / 2.0
        rmse = torch.sqrt(torch.mean((y_trues - mid_preds) ** 2)).item()
        
        interval_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
        
        print(f"PICP (Coverage):   {picp:.4f}")
        print(f"MPIW (Width):      ₹{mpiw:,.2f}")
        print(f"RMSE (Midpoint):   ₹{rmse:,.2f}")
        print(f"Interval Score:    {interval_score:,.2f}")

In [116]:
torch.manual_seed(54)
train_cross_attention_model(epochs=5)

Using device: cuda


CROSS-ATTENTION MODEL: BIKES
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹411,969.00
RMSE (Midpoint):   ₹159,851.97
Interval Score:    411,969.00

CROSS-ATTENTION MODEL: BOOKS
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹24,631.94
RMSE (Midpoint):   ₹10,614.55
Interval Score:    24,631.94

CROSS-ATTENTION MODEL: CARS
--------------------------------------------------
PICP (Coverage):   0.9677
MPIW (Width):      ₹26,821,416.00
RMSE (Midpoint):   ₹12,874,693.00
Interval Score:    26,862,670.00

CROSS-ATTENTION MODEL: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9310
MPIW (Width):      ₹128,615.91
RMSE (Midpoint):   ₹56,764.95
Interval Score:    133,201.47

CROSS-ATTENTION MODEL: FLAT
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹36,518,500.00
RMSE (Midpoint):   ₹11,941,645.00
Interv

In [71]:
torch.cuda.empty_cache()

In [16]:
class GatedMultimodalHead(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        
        self.img_transform = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.txt_transform = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        
        # It takes both original embeddings and outputs a value between 0 and 1
        self.gate_layer = nn.Sequential(
            nn.Linear(embed_dim * 2, 512),
            nn.Sigmoid() # Squashes output to (0, 1)
        )
        
        self.fc_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = x.float()
        
        # Split into Image and Text
        img_emb = x[:, :768]
        txt_emb = x[:, 768:]
        
        # Transform them individually
        img_feat = self.img_transform(img_emb)
        txt_feat = self.txt_transform(txt_emb)
        
        # Calculate the Gate
        # Gate = 1 means trust Image perfectly. Gate = 0 means trust Text perfectly.
        gate = self.gate_layer(x)
        
        # Fuse them using the Gate (This is the magic equation from the IJCAI review)
        fused_features = gate * img_feat + (1 - gate) * txt_feat
        
        # Predict
        return self.fc_head(fused_features)

In [147]:
def train_gated_multimodal_head_model(split_dir="split_embeddings", epochs=15):
    # Change to "cpu" or "cuda:1" if GPU 0 is blocked
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}\n")
    
    category_t_map = {
        "BIKES": 0.95, "BOOKS": 0.96, "CARS": 0.97, "CYCLE": 0.96,
        "FLAT": 0.97, "FRIDGES": 0.97, "GAMES": 0.94, "GAMESENTERTAINMENT": 0.96,
        "LAPTOP": 0.97, "MOBILE": 0.95, "PHONES": 0.95, "PRINTER": 0.97,
        "TV": 0.96, "WASHINGMACHINE": 0.94
    }
    
    train_files = [f for f in os.listdir(os.path.join(split_dir, 'train')) if f.endswith('.pt')]
    
    for file in train_files:
        cat_name = file.replace('combined_', '').replace('.pt', '').upper()
        train_path = os.path.join(split_dir, 'train', file)
        test_path = os.path.join(split_dir, 'test', file)
        
        if not os.path.exists(test_path): continue
            
        t_val = category_t_map.get(cat_name,0.95)
        alpha = 1.0 - t_val
        
        print(f"\nFEATURE-WISE GATING MODEL: {cat_name}\n{'-'*50}")
        
        # Load Data
        raw_train_data = torch.load(train_path, weights_only=False)
        cat_mu = raw_train_data['log_prices'].mean().item()
        cat_sigma = raw_train_data['log_prices'].std().item()
        
        train_dataset = ScaledRFPDataset(train_path, cat_mu, cat_sigma)
        test_dataset = ScaledRFPDataset(test_path, cat_mu, cat_sigma)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
        
        # Initialize Cross-Attention Model
        model = GatedMultimodalHead().to(device)
        optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

        # Train
        model.train()
        for epoch in range(epochs):
            for x, y in train_loader:
                x, y = x.to(device), y.to(device)
                
                optimizer.zero_grad()
                out = model(x)
                loss = tube_loss(y, out[:,0], out[:,1], t=t_val, r=0.5)
                loss.backward()
                optimizer.step()

        # Evaluate
        model.eval()
        lowers, uppers, ys = [], [], []
        
        with torch.no_grad():
            for x_batch, y_batch in test_loader:
                x_device = x_batch.to(device)
                
                out = model(x_device)
                l = torch.min(out[:,0], out[:,1])
                u = torch.max(out[:,0], out[:,1])
                
                # Inverse Transform to real Rupees
                l_real = torch.exp(l * cat_sigma + cat_mu)
                u_real = torch.exp(u * cat_sigma + cat_mu)
                y_true = torch.exp(y_batch.to(device) * cat_sigma + cat_mu)
                
                lowers.append(l_real)
                uppers.append(u_real)
                ys.append(y_true)
                
        l_preds = torch.cat(lowers)
        u_preds = torch.cat(uppers)
        y_trues = torch.cat(ys)
        
        # Metrics Calculations
        picp = ((y_trues >= l_preds) & (y_trues <= u_preds)).float().mean().item()
        mpiw = (u_preds - l_preds).mean().item()
        mid_preds = (l_preds + u_preds) / 2.0
        rmse = torch.sqrt(torch.mean((y_trues - mid_preds) ** 2)).item()
        
        # IJCAI Requirement: Interval Score
        interval_score = calculate_interval_score(y_trues, l_preds, u_preds, alpha=alpha)
        
        print(f"PICP (Coverage):   {picp:.4f}")
        print(f"MPIW (Width):      ₹{mpiw:,.2f}")
        print(f"RMSE (Midpoint):   ₹{rmse:,.2f}")
        print(f"Interval Score:    {interval_score:,.2f}")

In [148]:
torch.cuda.empty_cache()
torch.manual_seed(54)
train_gated_multimodal_head_model(epochs=5)

Using device: cuda


FEATURE-WISE GATING MODEL: BIKES
--------------------------------------------------
PICP (Coverage):   0.9773
MPIW (Width):      ₹273,046.16
RMSE (Midpoint):   ₹94,298.35
Interval Score:    304,721.88

FEATURE-WISE GATING MODEL: BOOKS
--------------------------------------------------
PICP (Coverage):   1.0000
MPIW (Width):      ₹33,125.79
RMSE (Midpoint):   ₹14,745.93
Interval Score:    33,125.79

FEATURE-WISE GATING MODEL: CARS
--------------------------------------------------
PICP (Coverage):   0.9677
MPIW (Width):      ₹30,317,466.00
RMSE (Midpoint):   ₹15,176,105.00
Interval Score:    30,382,960.00

FEATURE-WISE GATING MODEL: CYCLE
--------------------------------------------------
PICP (Coverage):   0.9655
MPIW (Width):      ₹28,762.07
RMSE (Midpoint):   ₹11,473.56
Interval Score:    35,017.14

FEATURE-WISE GATING MODEL: FLAT
--------------------------------------------------
PICP (Coverage):   0.9608
MPIW (Width):      ₹16,944,848.00
RMSE (Midpoint):   ₹4,1